[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/26_lora.ipynb)

# 🟠 Medium: LoRA (Low-Rank Adaptation)

Implement **LoRA** — parameter-efficient fine-tuning for large models.

$$h = W_0 x + \frac{\alpha}{r} B A x$$

### Signature
```python
class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank, alpha=1.0): ...
    def forward(self, x: Tensor) -> Tensor: ...
```

### Requirements
- `self.linear`: frozen `nn.Linear` (weight & bias `requires_grad=False`)
- `self.lora_A`: `nn.Parameter(rank, in_features)` — random init
- `self.lora_B`: `nn.Parameter(out_features, rank)` — **zero** init
- Scaling: `alpha / rank`

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.2 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn

In [17]:
# ✏️ YOUR IMPLEMENTATION HERE

class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank, alpha=1.0):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.linear.weight.requires_grad = False
        if self.linear.bias is not None:
            self.linear.bias.requires_grad = False

        la = torch.randn(rank, in_features)
        lb = torch.zeros(out_features, rank)
        self.lora_A = nn.Parameter(la)
        self.lora_B = nn.Parameter(lb)

        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        self.alpha = alpha


    def forward(self, x):
        base = self.linear(x)
        #print(base.shape)
        #print(self.lora_B.shape, self.lora_A.shape, x.shape)
        lora = self.lora_B @ self.lora_A @ x.transpose(1,0)
        lora = self.alpha / self.rank * lora
        return base + lora.transpose(1,0)

In [16]:
# 🧪 Debug
layer = LoRALinear(16, 8, rank=4)
x = torch.randn(2, 16)
print('Output:', layer(x).shape)
print('Trainable:', sum(p.numel() for p in layer.parameters() if p.requires_grad))
print('Total:    ', sum(p.numel() for p in layer.parameters()))

torch.Size([2, 8])
torch.Size([8, 4]) torch.Size([4, 16]) torch.Size([2, 16])
Output: torch.Size([2, 8])
Trainable: 96
Total:     232


In [18]:
# ✅ SUBMIT
from torch_judge import check
check('lora')


🧪 Testing: LoRA (Low-Rank Adaptation) (Medium)
──────────────────────────────────────────────────
  ✅ [1/5] Base weights frozen (1.1ms)
  ✅ [2/5] LoRA parameter shapes (0.4ms)
  ✅ [3/5] B=0 means output equals base (44.1ms)
  ✅ [4/5] Only LoRA params get gradients (25.7ms)
  ✅ [5/5] Forward computation (4.1ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (75.4ms total)
  Progress saved. Run status() to see your dashboard.

